# GANs (Generative Adversarial Networks) (generator vs discriminator)

_Deep Learning (CS230)_

**A forger and a detective compete; the forger gets so good its fakes look real.**

A GAN (Generative Adversarial Network) makes new, realistic data, like fake photos of people who don't exist.

     It has two networks playing a game. The generator is a forger making fakes. The discriminator is a detective judging real vs fake.

---

This notebook is a practice scaffold for the **GANs (Generative Adversarial Networks) (generator vs discriminator)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

G = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 4))   # noise -> fake
D = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 1))   # sample -> real?
bce = nn.BCEWithLogitsLoss()
optG = torch.optim.Adam(G.parameters(), lr=1e-3)
optD = torch.optim.Adam(D.parameters(), lr=1e-3)

real = torch.randn(32, 4)               # a batch of real samples
noise = torch.randn(32, 8)              # random seed for the generator

# 1) train discriminator: real -> 1, fake -> 0
fake = G(noise).detach()
optD.zero_grad()
lossD = bce(D(real), torch.ones(32, 1)) + bce(D(fake), torch.zeros(32, 1))
lossD.backward(); optD.step()

# 2) train generator: fool D into calling fakes real (label 1)
optG.zero_grad()
lossG = bce(D(G(noise)), torch.ones(32, 1))
lossG.backward(); optG.step()
print("lossD", round(lossD.item(), 3), "lossG", round(lossG.item(), 3))

## Visualize the data & results

_As a generator learns to mimic real digit-pixel statistics, does its output distance shrink toward the real data?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
real_mean = (digits.data / 16.0).mean()  # real target statistic = 0.263

fake = 0.0                               # generator starts producing dark images
trace = []
for step in range(41):
    trace.append(fake)
    fake += 0.1 * (real_mean - fake)     # learns toward the real data statistic

plt.plot(trace, color="#4ea1ff", label="mean pixel of fake batch")
plt.axhline(real_mean, color="#7ee787", label="real digit mean pixel (target)")
plt.title("Generator-vs-real distance and per-pixel mean error on load_digits")
plt.xlabel("step"); plt.ylabel("distance / error")
plt.legend()
plt.show()